# 04 · A differentiable neural network, trained by your engine

*Build Your Own Autograd: a Tiny Automatic Differentiation Engine*, notebook 4 of 4

You now have a working reverse-mode autodiff engine. In this notebook you use it the way PyTorch is used: build a small neural network out of `Value`s, write a loss, and let `backward()` compute the gradient of every parameter. **Nothing in this notebook derives a gradient by hand.** At the end you map each piece onto PyTorch and look at what a production framework adds.

One caution before you start. Your engine computes the same derivatives that a framework would on these small examples, but it is an educational model, **not production-equivalent**: it works on one number at a time in pure Python, so it is very slow, it supports only a handful of operations, and it has none of the numerical safeguards of a real framework.

## What you will build

- `neuron_forward(neuron, x)`: one unit, $\tanh(b + \sum_i w_i x_i)$;
- `layer_forward(layer, x)` and `mlp_forward(mlp, x)`: a layer of neurons and a multilayer perceptron;
- `mse_loss(predictions, targets)`: the mean squared error, as a `Value`;
- `train_step(model, X, y, learning_rate, loss_function)`: `zero_grad` → forward pass → loss → `backward()` → update.

**Learning objectives.** By the end you can:

1. build neurons, layers and a multilayer perceptron whose parameters are `Value`s;
2. write a loss as an expression of `Value`s, so that `backward()` reaches every parameter;
3. write and run a complete training loop, and judge it by its loss curve and accuracy;
4. map each piece onto PyTorch's `requires_grad`, `.backward()` and `.grad`, and explain what production frameworks add.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 0:** the core cells use only the Python standard library: `math` and `random` (imported in the setup cell) and `time`. The notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Two optional cells: one tagged `comparison` uses **NumPy** to check one gradient with an independent finite difference, and one tagged `visualization` uses **Matplotlib** to draw the decision boundary. Both skip themselves if the library is missing, and nothing depends on them.
- Nothing is downloaded, and every random number comes from a seeded `random.Random`.

## Setup

Run this cell first. It creates a seeded random generator, defines `practice`, the helper that checks each function you write against a reference, and imports `time`, which measures the training run.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import math
import random

rng = random.Random(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    if isinstance(want, (int, float)) and isinstance(got, (int, float)):
        return math.isclose(got, want, rel_tol=1e-9, abs_tol=atol)
    return got == want


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-9):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

## 📦 Recap: what you built earlier

The hidden cell below re-defines, under their usual names, the engine from notebooks 01–03: `central_difference`, `practice_with`, the `Value` class with `as_value` and its operations (`value_add`, `value_mul`, `value_pow`, `value_exp`, `value_log`, `value_tanh`, `value_relu`), `topological_order`, `backward_pass` (which `Value.backward()` calls), `zero_grad` and `gradient_check`. Run it; open it only if you want to look back at the code.

In [ ]:
# 📦 Recap: the engine you built in notebooks 01–03, under the usual names.

def central_difference(f, x, h=1e-6):
    """Estimate f'(x) from two nearby values of f: (f(x + h) - f(x - h)) / (2h)."""
    return (f(x + h) - f(x - h)) / (2 * h)


def practice_with(probe, name, yours, reference, cases, atol=1e-9):
    """Run practice on probe(yours) and probe(reference); return the function to use from now on.

    A probe wraps a function so that it returns plain numbers, lists or dicts
    that practice can compare. You get back your own function once it passes,
    otherwise the reference, exactly as with practice.
    """
    your_probe, reference_probe = probe(yours), probe(reference)
    chosen = practice(name, your_probe, reference_probe, cases, atol=atol)
    return yours if chosen is your_probe else reference


class Value:
    """A number that remembers how it was computed, so that gradients can flow back through it."""

    def __init__(self, data, children=(), op="", label=""):
        self.data = float(data)              # the number itself
        self.grad = 0.0                      # d(final output) / d(this value), filled in by the backward pass
        self._children = tuple(children)     # the Values this one was computed from
        self._op = op                        # the operation that made it, for printing
        self._backward = _no_gradient_to_pass_on   # replaced by the operation that creates this Value
        self.label = label                   # an optional name, for printing

    def __repr__(self):
        name = f"{self.label} | " if self.label else ""
        return f"Value({name}data={self.data:.6g}, grad={self.grad:.6g})"

    # Each operator hands the work to a module-level function, looked up when it runs.
    def __add__(self, other):        # self + other
        return value_add(self, as_value(other))

    def __radd__(self, other):       # other + self, when other is a plain number
        return value_add(as_value(other), self)

    def __mul__(self, other):        # self * other
        return value_mul(self, as_value(other))

    def __rmul__(self, other):       # other * self
        return value_mul(as_value(other), self)

    def __pow__(self, exponent):     # self ** exponent, for a constant int or float exponent
        if isinstance(exponent, Value):
            raise TypeError("the exponent must be a plain int or float, not a Value")
        return value_pow(self, exponent)

    # These four are built from the three above, so they need no rules of their own.
    def __neg__(self):               # -self
        return self * -1.0

    def __sub__(self, other):        # self - other
        return self + (-as_value(other))

    def __rsub__(self, other):       # other - self
        return as_value(other) + (-self)

    def __truediv__(self, other):    # self / other
        return self * as_value(other) ** -1

    def __rtruediv__(self, other):   # other / self
        return as_value(other) * self ** -1

    def exp(self):
        return value_exp(self)

    def log(self):
        return value_log(self)

    def tanh(self):
        return value_tanh(self)

    def relu(self):
        return value_relu(self)

    def backward(self):
        """Fill in .grad for every Value this one depends on."""
        backward_pass(self)


def _no_gradient_to_pass_on():
    """The _backward of a Value that no operation produced (an input): there is nothing to pass on."""


def as_value(x):
    """Wrap a plain number in a Value; leave a Value unchanged."""
    return x if isinstance(x, Value) else Value(x)


def value_add(a, b):
    out = Value(a.data + b.data, (a, b), "+")

    def _backward():
        a.grad += out.grad
        b.grad += out.grad

    out._backward = _backward
    return out


def value_mul(a, b):
    out = Value(a.data * b.data, (a, b), "*")

    def _backward():
        a.grad += b.data * out.grad
        b.grad += a.data * out.grad

    out._backward = _backward
    return out


def value_pow(a, exponent):
    out = Value(a.data ** exponent, (a,), f"**{exponent}")

    def _backward():
        a.grad += exponent * a.data ** (exponent - 1) * out.grad

    out._backward = _backward
    return out


def value_exp(a):
    out = Value(math.exp(a.data), (a,), "exp")

    def _backward():
        a.grad += out.data * out.grad

    out._backward = _backward
    return out


def value_log(a):
    out = Value(math.log(a.data), (a,), "log")

    def _backward():
        a.grad += out.grad / a.data

    out._backward = _backward
    return out


def value_tanh(a):
    out = Value(math.tanh(a.data), (a,), "tanh")

    def _backward():
        a.grad += (1.0 - out.data ** 2) * out.grad

    out._backward = _backward
    return out


def value_relu(a):
    out = Value(a.data if a.data > 0 else 0.0, (a,), "relu")

    def _backward():
        a.grad += (1.0 if a.data > 0 else 0.0) * out.grad

    out._backward = _backward
    return out


def topological_order(root):
    """Every node reachable from root, once each, children before parents (root last)."""
    order, visited = [], set()
    stack = [(root, False)]                 # (node, are its children already listed?)
    while stack:
        node, children_done = stack.pop()
        if children_done:
            order.append(node)
        elif node not in visited:
            visited.add(node)
            stack.append((node, True))      # come back to this node after its children
            for child in node._children:
                stack.append((child, False))
    return order


def backward_pass(root):
    """Seed root.grad = 1.0, then run every _backward in reverse topological order."""
    root.grad = 1.0
    for node in reversed(topological_order(root)):
        node._backward()


def zero_grad(params):
    """Reset the gradient of every Value in params to 0.0."""
    for param in params:
        param.grad = 0.0


def gradient_check(f, xs, h=1e-6):
    """Largest gap between backward() and central differences for f(*Values) at the point xs."""
    leaves = [Value(x) for x in xs]
    f(*leaves).backward()
    worst = 0.0
    for index in range(len(xs)):
        def along(t):
            shifted = list(xs)
            shifted[index] = t
            return f(*[Value(s) for s in shifted]).data
        worst = max(worst, abs(leaves[index].grad - central_difference(along, xs[index], h)))
    return worst

### How the checks work in this notebook

Your functions return `Value`s or change a model in place, so each check runs them through a **probe** that builds a fresh object from a seed and reports plain numbers:

- `neuron_probe`, `layer_probe` and `mlp_probe` build a `Neuron`, `Layer` or `MLP` from a seed and its sizes, run your forward function, back-propagate from the output (a fixed weighted sum of the outputs, for a layer) and report the outputs and every parameter's gradient;
- `loss_probe` turns a list of numbers into `Value` predictions and reports the loss and each prediction's gradient;
- `train_probe` builds a fresh `MLP(2, [3, 1])` from a seed, takes two or three steps with your `train_step` on four points, and reports the losses and the final parameters.

In [ ]:
def neuron_probe(forward):
    """Probe for neuron_forward: build a Neuron from a seed, run it, back-propagate, report numbers."""
    def run(seed, n_in, nonlinear, x):
        neuron = Neuron(n_in, random.Random(seed), nonlinear)
        out = forward(neuron, x)
        out.backward()
        return {"output": out.data, "parameter grads": [param.grad for param in neuron.parameters()]}
    return run


def _outputs_report(outputs, params):
    """Back-propagate a fixed weighted sum of the outputs; report them and every parameter's gradient."""
    total = Value(0.0)
    for weight, output in zip([1.0, -2.0, 3.0, -4.0], outputs):
        total = total + output * weight
    total.backward()
    return {"outputs": [output.data for output in outputs], "parameter grads": [param.grad for param in params]}


def layer_probe(forward):
    """Probe for layer_forward: a Layer built from a seed and its sizes."""
    def run(seed, n_in, n_out, nonlinear, x):
        layer = Layer(n_in, n_out, random.Random(seed), nonlinear)
        return _outputs_report(forward(layer, x), layer.parameters())
    return run


def mlp_probe(forward):
    """Probe for mlp_forward: an MLP built from a seed and its sizes."""
    def run(seed, n_in, sizes, x):
        mlp = MLP(n_in, sizes, random.Random(seed))
        return _outputs_report(forward(mlp, x), mlp.parameters())
    return run


def loss_probe(loss_function):
    """Probe for a loss: predictions given as numbers become Values; report the loss and their gradients."""
    def run(prediction_numbers, targets):
        predictions = [Value(number) for number in prediction_numbers]
        loss = loss_function(predictions, list(targets))
        loss.backward()
        return {"loss": loss.data, "prediction grads": [prediction.grad for prediction in predictions]}
    return run


TINY_X = [[0.5, -1.0], [-0.3, 0.8], [1.2, 0.1], [-0.7, -0.6]]
TINY_Y = [1.0, -1.0, 1.0, -1.0]


def train_probe(step_function):
    """Probe for train_step: a fresh MLP(2, [3, 1]) takes a few steps on four points."""
    def run(seed, learning_rate, steps):
        model = MLP(2, [3, 1], random.Random(seed))
        losses = [step_function(model, TINY_X, TINY_Y, learning_rate, mse_loss) for _ in range(steps)]
        return {"losses": losses, "parameters": [param.data for param in model.parameters()]}
    return run

## 1 · A neuron made of `Value`s

A neuron with $n$ inputs has $n$ weights and one bias, all `Value`s. These are the **parameters**: the numbers the training loop will adjust, like PyTorch tensors created with `requires_grad=True`. Given inputs $x_1, \dots, x_n$ (plain numbers, or `Value`s coming from a previous layer), it computes

$$\text{out} = \tanh\Big(b + \sum_{i=1}^{n} w_i x_i\Big),$$

or just the weighted sum for a **linear** unit (`nonlinear=False`), which the output layer uses. Because this is ordinary `Value` arithmetic, the graph and every backward rule come for free.

**Initialisation.** Each weight is drawn from a normal distribution with mean 0 and standard deviation $1/\sqrt{n}$, and the bias starts at 0. Scaling by $1/\sqrt{n}$ keeps the weighted sum near unit size whatever $n$ is (for inputs of about unit size), so tanh units start in their responsive range rather than saturated; notebook 02's 🧪 showed how little gradient a saturated tanh passes back. The random weights also make the neurons of a layer different from one another. If they all started equal, they would all receive the same gradient and stay equal for ever.

In [ ]:
class Neuron:
    """One unit: a weighted sum of its inputs plus a bias, passed through tanh unless nonlinear=False."""

    def __init__(self, n_in, rng, nonlinear=True):
        spread = 1.0 / math.sqrt(n_in)            # keeps b + Σ wᵢxᵢ near unit size for unit-size inputs
        self.w = [Value(rng.gauss(0.0, spread), label=f"w{index}") for index in range(n_in)]
        self.b = Value(0.0, label="b")
        self.nonlinear = nonlinear

    def __call__(self, x):
        return neuron_forward(self, x)

    def parameters(self):
        return [*self.w, self.b]

    def __repr__(self):
        return f"Neuron({len(self.w)} inputs, {'tanh' if self.nonlinear else 'linear'})"


demo_neuron = Neuron(3, random.Random(1))
print(demo_neuron, [round(param.data, 4) for param in demo_neuron.parameters()])

### ✍️ Your turn: `neuron_forward`

Write `neuron_forward(neuron, x)`:

- **inputs:** `neuron`, a `Neuron` (its weights are `neuron.w`, its bias `neuron.b`, and `neuron.nonlinear` says whether to apply tanh); `x`, a list of `len(neuron.w)` numbers or `Value`s;
- **output:** one `Value`, $b + \sum_i w_i x_i$, passed through `.tanh()` if `neuron.nonlinear` is true.

Hints: start a running total at `neuron.b` and add `weight * value` for each pair from `zip(neuron.w, x)`. Raise a `ValueError` if `len(x)` is not `len(neuron.w)`: `zip` would otherwise silently ignore the extra inputs.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def neuron_forward_reference(neuron, x):
    if len(x) != len(neuron.w):
        raise ValueError(f"this neuron takes {len(neuron.w)} inputs, not {len(x)}")
    total = neuron.b
    for weight, value in zip(neuron.w, x):
        total = total + weight * value
    return total.tanh() if neuron.nonlinear else total

In [ ]:
def neuron_forward(neuron, x):
    """b + Σ wᵢ·xᵢ as a Value, through tanh when neuron.nonlinear is true."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
neuron_forward = practice_with(neuron_probe, "neuron_forward", neuron_forward, neuron_forward_reference, cases=[
    (1, 3, True, [0.5, -1.0, 2.0]),
    (2, 2, False, [1.0, -1.0]),
    (3, 1, True, [0.0]),                   # edge case: one input, equal to 0
])
print(demo_neuron([0.5, -1.0, 2.0]))

**Expected behaviour.** ✅, then one `Value` between $-1$ and $1$, the output of a tanh unit. The probe also back-propagated from each output and compared every parameter's gradient: for a tanh unit, $\partial\,\text{out}/\partial w_i = (1 - \text{out}^2)\, x_i$, and the bias gets $1 - \text{out}^2$. You never wrote those formulas; the engine applied them.

## 2 · A layer

A **layer** is a list of neurons that all read the same inputs; its output is the list of their outputs, one `Value` per neuron. A layer with $n_\text{in}$ inputs and $n_\text{out}$ neurons has $n_\text{out}(n_\text{in} + 1)$ parameters.

In [ ]:
class Layer:
    """n_out neurons that all read the same inputs."""

    def __init__(self, n_in, n_out, rng, nonlinear=True):
        self.neurons = [Neuron(n_in, rng, nonlinear) for _ in range(n_out)]

    def __call__(self, x):
        return layer_forward(self, x)

    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

    def __repr__(self):
        kind = "tanh" if self.neurons[0].nonlinear else "linear"
        return f"Layer({len(self.neurons[0].w)} inputs → {len(self.neurons)} neurons, {kind})"


demo_layer = Layer(2, 3, random.Random(4))
print(demo_layer, "with", len(demo_layer.parameters()), "parameters")

### ✍️ Your turn: `layer_forward`

Write `layer_forward(layer, x)`:

- **inputs:** a `Layer` (its neurons are `layer.neurons`) and the input list `x`;
- **output:** a list with one `Value` per neuron, in the order of `layer.neurons`. Always return a list, even for a layer of one neuron.

Hint: calling a neuron, `neuron(x)`, runs your `neuron_forward`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def layer_forward_reference(layer, x):
    outputs = []
    for neuron in layer.neurons:
        outputs.append(neuron(x))
    return outputs

In [ ]:
def layer_forward(layer, x):
    """A list with one output Value per neuron of the layer."""
    # ✍️ Your code here (one to three lines)
    raise NotImplementedError

In [ ]:
layer_forward = practice_with(layer_probe, "layer_forward", layer_forward, layer_forward_reference, cases=[
    (4, 2, 3, True, [0.3, -0.8]),
    (5, 3, 2, False, [1.0, 0.5, -0.5]),
    (6, 2, 1, True, [0.0, 1.0]),            # edge case: one neuron still gives a list
])
print(demo_layer([0.3, -0.8]))

**Expected behaviour.** ✅, then a list of three `Value`s, one per neuron. Each neuron saw the same two inputs but has its own weights, so the outputs differ.

## 3 · A multilayer perceptron

A **multilayer perceptron** (MLP) chains layers: each layer's output list is the next layer's input. `MLP(n_in, sizes, rng)` builds one layer per entry of `sizes`. The hidden layers use tanh, and the last layer is linear, so the network's output is any real number: a **score** whose sign gives the class. `MLP(2, [8, 1], rng)` maps 2 inputs through 8 tanh units to 1 output and has $8 \times (2 + 1) + 1 \times (8 + 1) = 33$ parameters. The network returns a list of outputs, so its single score is `model(x)[0]`.

In [ ]:
class MLP:
    """A stack of layers: tanh hidden layers and a linear output layer."""

    def __init__(self, n_in, sizes, rng):
        widths = [n_in, *sizes]
        self.layers = []
        for index in range(len(sizes)):
            is_output = index == len(sizes) - 1
            self.layers.append(Layer(widths[index], widths[index + 1], rng, nonlinear=not is_output))

    def __call__(self, x):
        return mlp_forward(self, x)

    def parameters(self):
        params = []
        for layer in self.layers:
            params.extend(layer.parameters())
        return params

    def __repr__(self):
        widths = [len(self.layers[0].neurons[0].w)] + [len(layer.neurons) for layer in self.layers]
        return "MLP(" + " → ".join(str(width) for width in widths) + ")"


demo_mlp = MLP(2, [8, 1], random.Random(7))
print(demo_mlp, "with", len(demo_mlp.parameters()), "parameters")

### ✍️ Your turn: `mlp_forward`

Write `mlp_forward(mlp, x)`:

- **inputs:** an `MLP` (its layers are `mlp.layers`) and the input list `x`;
- **output:** the list of `Value`s that the last layer produces.

Hint: a loop that replaces the current list with `layer(current)`; calling a layer runs your `layer_forward`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mlp_forward_reference(mlp, x):
    activations = list(x)
    for layer in mlp.layers:
        activations = layer(activations)
    return activations

In [ ]:
def mlp_forward(mlp, x):
    """Feed x through every layer in turn; return the last layer's list of Values."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
mlp_forward = practice_with(mlp_probe, "mlp_forward", mlp_forward, mlp_forward_reference, cases=[
    (8, 2, [4, 1], [0.3, -0.8]),
    (9, 3, [4, 4, 2], [1.0, 0.5, -0.5]),
    (10, 2, [1], [2.0, -1.0]),              # edge case: no hidden layer, a single linear unit
])
score = demo_mlp([0.5, -0.2])[0]
print("score:", score, "  nodes in its graph:", len(topological_order(score)))

**Expected behaviour.** ✅, then one score and the size of the graph behind it: about a hundred nodes for a single point passing through 33 parameters. Every one of them will receive a gradient when you call `backward()`.

## 4 · Data: two interleaved half-moons

A classic small problem that no straight line solves completely: two crescent-shaped clouds of points that interlock. `make_moons` generates them in pure Python with a seeded `random.Random`, so the data are the same everywhere. Points on the upper moon have label $+1$ and points on the lower moon $-1$. Sixty points are enough to learn the shape and few enough for a pure-Python engine. `text_plot` draws the points as characters (`+` for $+1$, `o` for $-1$); given a model, it also shades the region where the model predicts $+1$ with `·`.

In [ ]:
def make_moons(n_points, noise, rng):
    """Two interleaved half circles in 2-D, labelled +1 and -1, centred near the origin."""
    X, y = [], []
    for index in range(n_points):
        angle = math.pi * rng.random()
        if index % 2 == 0:                       # the upper moon
            u, v, label = math.cos(angle), math.sin(angle), 1.0
        else:                                    # the lower moon, shifted right and down
            u, v, label = 1.0 - math.cos(angle), 0.5 - math.sin(angle), -1.0
        u += rng.gauss(0.0, noise)
        v += rng.gauss(0.0, noise)
        X.append([u - 0.5, 2.0 * (v - 0.25)])    # centre the cloud and stretch it vertically
        y.append(label)
    return X, y


def text_plot(X, y, model=None, width=56, height=18):
    """Draw the points as '+' (label +1) and 'o' (label -1); with a model, shade its +1 region with '·'."""
    us, vs = [point[0] for point in X], [point[1] for point in X]
    left, right, bottom, top = min(us) - 0.2, max(us) + 0.2, min(vs) - 0.2, max(vs) + 0.2
    grid = [[" "] * width for _ in range(height)]
    if model is not None:
        for row in range(height):
            for column in range(width):
                u = left + (column + 0.5) * (right - left) / width
                v = top - (row + 0.5) * (top - bottom) / height
                if model([u, v])[0].data > 0:
                    grid[row][column] = "·"
    for (u, v), label in zip(X, y):
        column = min(width - 1, int((u - left) / (right - left) * width))
        row = min(height - 1, int((top - v) / (top - bottom) * height))
        grid[row][column] = "+" if label > 0 else "o"
    print("\n".join("".join(line) for line in grid))


X_train, y_train = make_moons(60, noise=0.1, rng=random.Random(42))
print(len(X_train), "points; the first:", [round(value, 3) for value in X_train[0]], "label", y_train[0])
text_plot(X_train, y_train)

## 5 · A loss the engine can differentiate

Training needs one number that is small when the predictions are good: the **loss**. With targets $\pm 1$ and real-valued scores $s_i$, the mean squared error

$$L = \frac{1}{N} \sum_{i=1}^{N} (s_i - y_i)^2$$

works well for this problem. Because it is built from `Value` operations, `L.backward()` reaches every prediction, with $\partial L / \partial s_i = 2(s_i - y_i)/N$, and through the predictions every parameter.

In [ ]:
print("by hand, for scores 0.8 and -0.3 with targets +1 and -1:", ((0.8 - 1.0) ** 2 + (-0.3 + 1.0) ** 2) / 2)

### ✍️ Your turn: `mse_loss`

Write `mse_loss(predictions, targets)`:

- **inputs:** `predictions`, a list of $N$ `Value`s; `targets`, a list of $N$ numbers;
- **output:** one `Value`, the mean of `(prediction - target) ** 2` over the $N$ pairs.

Hints: start from `Value(0.0)`, add each squared error, then divide by $N$ (or multiply by $1/N$). Raise a `ValueError` if the lists are empty or differ in length.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def mse_loss_reference(predictions, targets):
    if not predictions or len(predictions) != len(targets):
        raise ValueError("need the same, non-zero number of predictions and targets")
    total = Value(0.0)
    for prediction, target in zip(predictions, targets):
        error = prediction - target
        total = total + error * error
    return total * (1.0 / len(predictions))

In [ ]:
def mse_loss(predictions, targets):
    """The mean squared error of a list of Values against a list of numbers, as a Value."""
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
mse_loss = practice_with(loss_probe, "mse_loss", mse_loss, mse_loss_reference, cases=[
    ([0.8, -0.3], [1.0, -1.0]),
    ([0.5, 0.5, -2.0], [1.0, -1.0, -1.0]),
    ([1.0], [1.0]),                         # edge case: a perfect prediction has loss 0 and gradient 0
])
print(mse_loss([Value(0.8), Value(-0.3)], [1.0, -1.0]))

**Expected behaviour.** ✅, then a loss of $0.265$, matching the hand calculation. In the probe, the predictions' gradients are $2(s_i - y_i)/N$: $-0.2$ and $0.7$ for the first case.

## 6 · One training step

A training step is five lines, always in this order:

1. `zero_grad(params)`: forget the previous step's gradients;
2. the forward pass, `[model(point)[0] for point in X]`, which builds a new graph;
3. the loss, `loss_function(predictions, y)`;
4. `loss.backward()`: now every parameter's `grad` holds $\partial L / \partial\,\text{parameter}$;
5. the update, `param.data -= learning_rate * param.grad` for every parameter: a small step downhill.

The update changes `param.data`, never `param` itself: `param = param - …` would only build a new graph node and leave the model untouched. The step returns `loss.data`, the loss **before** the update, so that the caller can record a learning curve. The demo runs steps 1 to 4 by hand on five points.

In [ ]:
zero_grad(demo_mlp.parameters())
demo_loss = mse_loss([demo_mlp(point)[0] for point in X_train[:5]], y_train[:5])
demo_loss.backward()
first = demo_mlp.parameters()[0]
print(f"loss on 5 points {demo_loss.data:.4f}; the first weight is {first.data:+.4f} with gradient {first.grad:+.4f}")

### ✍️ Your turn: `train_step`

Write `train_step(model, X, y, learning_rate, loss_function)`:

- **inputs:** `model`, an `MLP` with one output; `X`, a list of points; `y`, their targets; `learning_rate`, a float; `loss_function`, such as `mse_loss`;
- **effect and output:** performs the five steps above once and returns the loss **before** the update, as a float.

Hints: get `params = model.parameters()` once, at the top. The check takes two or three steps in a row on a fresh network, so a missing `zero_grad` shows up as a mismatch.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def train_step_reference(model, X, y, learning_rate, loss_function):
    params = model.parameters()
    zero_grad(params)
    predictions = [model(point)[0] for point in X]
    loss = loss_function(predictions, y)
    loss.backward()
    for param in params:
        param.data -= learning_rate * param.grad
    return loss.data

In [ ]:
def train_step(model, X, y, learning_rate, loss_function):
    """zero_grad → forward → loss → backward → update; return the loss before the update."""
    # ✍️ Your code here (about eight lines)
    raise NotImplementedError

In [ ]:
train_step = practice_with(train_probe, "train_step", train_step, train_step_reference, cases=[
    (0, 0.1, 2),
    (1, 0.5, 3),
    (2, 0.0, 2),                            # edge case: a learning rate of 0 changes nothing
])


def accuracy(model, X, y):
    """The fraction of points whose score has the same sign as their label."""
    correct = 0
    for point, label in zip(X, y):
        correct += (model(point)[0].data > 0) == (label > 0)
    return correct / len(y)

**Expected behaviour.** ✅ on all three cases. With a learning rate of 0 the two recorded losses are equal and the parameters keep their initial values, which is a useful sanity check of any training loop. `accuracy`, defined in the same cell, counts the points whose score has the right sign.

## 🧪 Experiments

### 🧪 Try it: the learning rate

Forty steps from the same starting network with different learning rates. Each run takes a fraction of a second.

In [ ]:
learning_rate = 0.2   # try 0.02, then 0.6, and run this cell again
trial_model = MLP(2, [8, 1], random.Random(3))
try:
    trial_losses = [train_step(trial_model, X_train, y_train, learning_rate, mse_loss) for _ in range(40)]
except OverflowError:
    trial_losses = [float("nan")]
start_loss, end_loss = trial_losses[0], trial_losses[-1]
if math.isfinite(end_loss) and end_loss < start_loss:
    print(f"learning rate {learning_rate}: loss {start_loss:.3f} -> {end_loss:.3f} after 40 steps, "
          f"accuracy {accuracy(trial_model, X_train, y_train):.2f}")
else:
    print(f"learning rate {learning_rate}: the loss did not fall (it ended at {end_loss:.3g}): the steps are too large")

**What to notice.** With $0.02$ the loss falls more slowly, to about $0.35$ after forty steps. With $0.2$ it falls from about $2.2$ to under $0.3$. With $0.6$ the steps overshoot: the loss explodes to an astronomically large number within forty steps. Larger still, and Python's floats can overflow to `inf` or raise `OverflowError`. Gradient descent only works when each step is small enough for the gradient to stay a good guide.

### 🧪 Try it: does the hidden layer matter?

`sizes = [1]` is a network with no hidden layer: a single linear unit, whose decision boundary is a straight line. Each run trains for 100 steps.

In [ ]:
sizes = [8, 1]   # try [1] (a straight line) and [16, 1], then run this cell again
width_model = MLP(2, sizes, random.Random(3))
start = time.perf_counter()
width_losses = [train_step(width_model, X_train, y_train, 0.2, mse_loss) for _ in range(100)]
print(f"{width_model} with {len(width_model.parameters())} parameters: loss {width_losses[0]:.3f} -> {width_losses[-1]:.3f}, "
      f"accuracy {accuracy(width_model, X_train, y_train):.2f}, {time.perf_counter() - start:.1f} s")

**What to notice.** The straight line gets stuck near a loss of $0.31$ and 88% accuracy: some points of each moon lie on the wrong side of any line. Eight tanh units reach a loss of about $0.17$ in the same 100 steps, and sixteen go a little lower, at about twice the time per step. The time grows with the number of graph nodes, which grows with the number of parameters.

## 7 · Put it together: train the network

Now the real run: a fresh `MLP(2, [8, 1])` with its own seed, 200 full-batch steps of gradient descent with a learning rate of $0.2$, the loss printed every 25 steps, and the learned regions drawn as text at the end. On CPython this takes about two seconds; in the browser it takes several times longer.

The `assert`s use floors measured with a margin. Over 18 combinations of six network seeds and three data seeds, the final loss was at most 15% of the first and the training accuracy at least 91.7%. The asserts require a final loss below 30% of the first and an accuracy of at least 85%.

In [ ]:
model = MLP(2, [8, 1], random.Random(3))
EPOCHS, LEARNING_RATE = 200, 0.2
start_accuracy = accuracy(model, X_train, y_train)
history = []
start = time.perf_counter()
for epoch in range(EPOCHS):
    history.append(train_step(model, X_train, y_train, LEARNING_RATE, mse_loss))
    if epoch % 25 == 0 or epoch == EPOCHS - 1:
        print(f"epoch {epoch:>3}: loss {history[-1]:.4f}")
seconds = time.perf_counter() - start
final_accuracy = accuracy(model, X_train, y_train)
print(f"\n{EPOCHS} steps in {seconds:.1f} s; training accuracy {start_accuracy:.2f} before and {final_accuracy:.2f} after\n")
text_plot(X_train, y_train, model)

assert history[-1] <= 0.3 * history[0], "the loss should fall to below 30% of its starting value"
assert final_accuracy >= 0.85, "the network should classify at least 85% of the training points correctly"
assert gradient_check(lambda *ws: (ws[0] * 0.5 + ws[1]).tanh() * ws[2], [0.3, -0.2, 1.4]) < 1e-6, "the engine's gradients should still check out"

**What to notice.** The loss falls quickly at first and then slowly, from about $2.19$ to about $0.11$, and the training accuracy ends around 97%. In the text plot, the `·` region (where the network predicts $+1$) covers the upper moon, and its edge bends to follow the gap between the two moons, which a single linear unit cannot do. This is training accuracy on 60 generated points, so it says the engine and the loop work; it is not an estimate of how well the network would do on new data, which needs a held-out set (the Model Evaluation course covers this).

**Optional comparison.** The next cell is tagged `comparison`. It copies the trained weights into NumPy arrays, recomputes the loss with matrix products, an implementation that shares no code with your engine, and estimates the gradient of one weight with a central difference on that NumPy loss. It needs NumPy, which JupyterLite, the course server and Colab provide; nothing later depends on it.

In [ ]:
# Optional: needs NumPy. Nothing later in the notebook depends on this cell.
try:
    import numpy as np
except ImportError:
    np = None
    print("NumPy is not available; skipping the comparison.")

if np is not None:
    weights = [np.array([[w.data for w in neuron.w] for neuron in layer.neurons]) for layer in model.layers]
    biases = [np.array([neuron.b.data for neuron in layer.neurons]) for layer in model.layers]
    X_array, y_array = np.array(X_train), np.array(y_train)

    def numpy_loss(weights):
        activations = X_array
        for index, (W, b) in enumerate(zip(weights, biases)):
            activations = activations @ W.T + b
            if index < len(weights) - 1:
                activations = np.tanh(activations)
        return float(np.mean((activations[:, 0] - y_array) ** 2))

    step = 1e-6
    ahead = [W.copy() for W in weights]
    behind = [W.copy() for W in weights]
    ahead[0][0, 1] += step                      # first layer, neuron 0, weight on input 1
    behind[0][0, 1] -= step
    numpy_estimate = (numpy_loss(ahead) - numpy_loss(behind)) / (2 * step)

    zero_grad(model.parameters())
    mse_loss([model(point)[0] for point in X_train], y_train).backward()
    engine_gradient = model.layers[0].neurons[0].w[1].grad
    print(f"your engine: {engine_gradient:+.10f}   NumPy central difference: {numpy_estimate:+.10f}")
    assert abs(engine_gradient - numpy_estimate) < 1e-6, "the two gradients should agree"

**Optional picture.** The next cell is tagged `visualization`. It draws the decision boundary of the trained network with Matplotlib, if Matplotlib is installed. Nothing later depends on it.

In [ ]:
# Optional: needs Matplotlib. Nothing later in the notebook depends on this cell.
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("Matplotlib is not available; the text plot above shows the same regions.")

if plt is not None:
    us = [point[0] for point in X_train]
    vs = [point[1] for point in X_train]
    grid_u = [min(us) - 0.3 + (max(us) - min(us) + 0.6) * i / 39 for i in range(40)]
    grid_v = [min(vs) - 0.3 + (max(vs) - min(vs) + 0.6) * j / 29 for j in range(30)]
    scores = [[model([u, v])[0].data for u in grid_u] for v in grid_v]
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ax.contourf(grid_u, grid_v, scores, levels=[-1e9, 0.0, 1e9], colors=["#f3d3c4", "#cfe0f3"])
    ax.contour(grid_u, grid_v, scores, levels=[0.0], colors="0.2", linewidths=1)
    ax.scatter([u for u, label in zip(us, y_train) if label > 0], [v for v, label in zip(vs, y_train) if label > 0],
               marker="+", color="#1f4e8c", label="label +1")
    ax.scatter([u for u, label in zip(us, y_train) if label < 0], [v for v, label in zip(vs, y_train) if label < 0],
               marker="o", facecolors="none", edgecolors="#a33b20", label="label -1")
    ax.set_title("Decision boundary of the MLP trained by your engine")
    ax.legend(loc="upper right")
    plt.show()

### From your engine to PyTorch

Every piece you built has a direct counterpart in PyTorch:

| In this course | In PyTorch |
|---|---|
| a `Value` that training updates, such as a weight | a tensor created with `requires_grad=True`, or a parameter inside `nn.Linear` |
| the graph that every operation records | the same idea: each result has a `grad_fn` that links back to its inputs |
| each operation's `_backward` closure | the backward formula of that operation's `grad_fn` |
| `topological_order` and `backward_pass` | the autograd engine that `loss.backward()` runs |
| `param.grad` | `param.grad` |
| `zero_grad(params)` | `optimizer.zero_grad()` |
| `param.data -= learning_rate * param.grad` | `optimizer.step()` with `torch.optim.SGD`, or an update inside `with torch.no_grad():` |
| `Neuron`, `Layer` and `MLP` | `nn.Linear`, `nn.Tanh`, `nn.Sequential` and `nn.Module` |

The training loop above would look like this in PyTorch. It is shown for comparison and is not run here, because this course needs only the standard library:

```python
import torch
from torch import nn

model = nn.Sequential(nn.Linear(2, 8), nn.Tanh(), nn.Linear(8, 1))
optimizer = torch.optim.SGD(model.parameters(), lr=0.2)
X, y = torch.tensor(X_train), torch.tensor(y_train)
for epoch in range(200):
    optimizer.zero_grad()                              # zero_grad(params)
    loss = ((model(X).squeeze(1) - y) ** 2).mean()     # forward pass and mse_loss
    loss.backward()                                    # backward_pass(loss)
    optimizer.step()                                   # param.data -= lr * param.grad
```

**What production frameworks add.** Your engine is not production-equivalent. Real frameworks add, among much else:

- **tensors:** one graph node holds a whole array, so one backward formula handles millions of numbers at once, while your engine creates a node for every single multiplication (about 4,700 per training step here);
- **fast kernels and hardware:** operations run as compiled, vectorised code on CPUs, GPUs and other accelerators, not in the Python interpreter;
- **hundreds of operations**, each with a tested derivative, and numerically careful combinations such as a fused log-softmax and cross-entropy;
- **memory management:** by default PyTorch frees a graph's saved values once `backward()` has used them;
- **more kinds of differentiation:** gradients of gradients, forward mode, detection of in-place changes that would corrupt a gradient, mixed precision and compilation.

## Exercises

### Exercise 1 · The hinge loss

The hinge loss, used by support vector machines,

$$L = \frac{1}{N} \sum_{i=1}^{N} \max(0,\ 1 - y_i s_i),$$

is zero for a point classified correctly with a margin of at least 1 ($y_i s_i \ge 1$) and grows linearly otherwise. Write `hinge_loss(predictions, targets)` with the `relu()` method from notebook 02, then run the cell after the check to train a network with it.

In [ ]:
def hinge_loss_reference(predictions, targets):
    if not predictions or len(predictions) != len(targets):
        raise ValueError("need the same, non-zero number of predictions and targets")
    total = Value(0.0)
    for prediction, target in zip(predictions, targets):
        total = total + (1.0 - prediction * target).relu()
    return total * (1.0 / len(predictions))

In [ ]:
def hinge_loss(predictions, targets):
    """The mean of max(0, 1 - target·prediction), as a Value."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
hinge_loss = practice_with(loss_probe, "hinge_loss", hinge_loss, hinge_loss_reference, cases=[
    ([0.8, -0.3], [1.0, -1.0]),
    ([2.0, -1.5, 0.2], [1.0, -1.0, -1.0]),
    ([1.5], [1.0]),                         # a margin above 1: no loss and no gradient
])
hinge_model = MLP(2, [8, 1], random.Random(3))
hinge_history = [train_step(hinge_model, X_train, y_train, 0.1, hinge_loss) for _ in range(100)]
print(f"hinge loss {hinge_history[0]:.3f} -> {hinge_history[-1]:.3f}; accuracy {accuracy(hinge_model, X_train, y_train):.2f}")

### Exercise 2 · Weight decay

Adding a penalty $\lambda \sum_j p_j^2$ over all parameters to the loss pulls every weight towards zero; this is called L2 regularisation or **weight decay**. Write `l2_penalty(params, strength)`, which returns $\text{strength} \times \sum_j p_j^2$ as a `Value`. Then try `mse_loss(predictions, y) + l2_penalty(model.parameters(), 1e-3)` as a loss: you will need a small loss function that adds the two, because `train_step` passes only the predictions and the targets.

In [ ]:
def l2_penalty_reference(params, strength):
    total = Value(0.0)
    for param in params:
        total = total + param * param
    return total * strength


def penalty_probe(penalty_function):
    """Probe for l2_penalty: parameters given as numbers; report the penalty and their gradients."""
    def run(numbers, strength):
        params = [Value(number) for number in numbers]
        penalty = penalty_function(params, strength)
        penalty.backward()
        return {"penalty": penalty.data, "grads": [param.grad for param in params]}
    return run

In [ ]:
def l2_penalty(params, strength):
    """strength × the sum of the squares of the parameters, as a Value."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
l2_penalty = practice_with(penalty_probe, "l2_penalty", l2_penalty, l2_penalty_reference, cases=[
    ([0.5, -1.0, 2.0], 0.01),
    ([3.0], 0.5),
    ([], 0.1),                              # edge case: no parameters, no penalty
])

### Exercise 3 · Open questions

1. Count the `Value`s that one training step creates, with `len(topological_order(loss))`. How does the count grow with the number of points and with the width of the hidden layer, and what does that say about the speed of a scalar engine?
2. Every step above uses all 60 points (full-batch gradient descent). Sketch how to change `train_step` to use a random mini-batch of 10 points, with a seeded `random.Random`. What would you expect to happen to the time per step and to the shape of the loss curve?
3. The capstone asks you to add momentum: $v \leftarrow \beta v + \nabla L$, then $p \leftarrow p - \eta v$. Where would you store the velocity $v$ of each parameter, and why must it **not** be reset by `zero_grad`?

## Common errors and debugging

| Symptom | Likely cause | Fix |
|---|---|---|
| the loss does not change at all | the update assigns to `param` instead of `param.data`, or the learning rate is 0 | `param.data -= learning_rate * param.grad` |
| the loss rises steadily | the update has the wrong sign | step **against** the gradient: `-=` |
| the loss swings wildly or explodes, or `OverflowError` | the learning rate is too large | lower it; 0.05 to 0.3 works here |
| the steps get larger and larger | `zero_grad` is missing, so gradients pile up across steps | call `zero_grad(params)` at the start of every step |
| `TypeError: unsupported operand type(s) for -: 'list' and 'float'` | `model(x)` returns a list | take the score with `model(x)[0]` |
| every gradient is 0 | the predictions were built from `.data` floats, so the loss is not connected to the parameters | build the loss from `Value`s |
| `ValueError: this neuron takes 2 inputs, not 3` | a point with the wrong number of features | check the data |
| training is very slow | pure-Python scalar autodiff creates thousands of nodes per step | use fewer points, units or steps; real frameworks use tensors |

## Summary

- A neuron, a layer and an MLP are a few lines each on top of `Value`: their parameters are `Value`s, and their forward passes are ordinary arithmetic that records the graph.
- A loss built from `Value`s connects every parameter to one number, so a single `loss.backward()` fills in every parameter's gradient.
- A training step is always: `zero_grad` → forward pass → loss → `backward()` → `param.data -= learning_rate * param.grad`.
- Your engine trained a 33-parameter network to separate two interleaved moons, and its gradients agree with central differences and with an independent NumPy computation.
- PyTorch's `requires_grad`, `.backward()`, `.grad`, `optimizer.zero_grad()` and `optimizer.step()` are the same ideas, on tensors, with fast kernels, GPUs and many more operations. The educational engine is not production-equivalent.

You built `neuron_forward`, `layer_forward`, `mlp_forward`, `mse_loss` and `train_step`.

## Next

This is the last notebook of the course. The capstone asks you to extend your engine with a new operation and a momentum optimiser. After that, **Base Model Training** takes the same ideas to whole arrays (tensors) and to a complete training workflow, with data splits, validation and checkpoints. Every time you call `backward()` there, you will know what happens inside.